In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime
import time
from datetime import timedelta

# --- API CONFIGURATION AND CREDENTIALS ---
API_KEY = os.environ.get("ALPHA_VANTAGE_API_KEY") 
TICKERS = ['MSFT', 'AAPL', 'NVDA', 'GOOGL', 'AMZN']
DB_USER = "airflow"
DB_PASS = "airflow"
DB_HOST = "postgres"
DB_NAME = "airflow"
SCHEMA_NAME = "raw"
TABLE_NAME = "stock_prices_alphavantage"
INTERVAL = "DAILY"
FUNCTION = "TIME_SERIES_DAILY"

df_shape = pd.DataFrame()

for TI in TICKERS:
    # URL for getting data
    url = f'https://www.alphavantage.co/query?function={FUNCTION}&symbol={TI}&interval={INTERVAL}&apikey={API_KEY}'
    r = requests.get(url)
    data = r.json()

    # Extract time series data
    time_series_data = data.get("Time Series (Daily)", {})

    # Get yesterday's date in the format "YYYY-MM-DD"
    d1_str = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

    # Check if data for d1_str exists
    if d1_str in time_series_data:
        time_series_d1 = time_series_data[d1_str]

        # Convert the data to a DataFrame
        df_response = pd.DataFrame.from_dict(time_series_d1, orient='index').T

        # Rename columns to remove prefixes
        df_response.columns = [str(col).split('. ')[-1] for col in df_response.columns]

        # Add the ticker column
        df_response['ticker'] = TI

        # Add the date column
        df_response['date'] = d1_str

        # Concatenate to the main dataframe
        df_shape = pd.concat([df_shape, df_response], ignore_index=True)

    time.sleep(5)  # To avoid hitting the API rate limit


,open,high,low,close,volume,ticker,date
0,510.2250,515.2820,506.0000,513.5700,14684300,MSFT,2025-10-14
1,246.6000,248.8450,244.7000,247.7700,35477986,AAPL,2025-10-14
2,184.7700,184.8000,179.7000,180.0300,205641380,NVDA,2025-10-14
3,241.2300,247.1200,240.5100,245.4500,22111572,GOOGL,2025-10-14
4,215.5550,219.3200,212.6000,216.3900,45665580,AMZN,2025-10-14


In [15]:
df_shape.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   open    5 non-null      object
 1   high    5 non-null      object
 2   low     5 non-null      object
 3   close   5 non-null      object
 4   volume  5 non-null      object
 5   ticker  5 non-null      object
 6   date    5 non-null      object
dtypes: object(7)
memory usage: 412.0+ bytes
